# Part 5 — ETL/ELT, Orchestration, Data Quality

*CinemaStream — The Forward Deployed Engineer's Handbook*

---

In [ ]:
# ── CinemaStream: one-time setup ──────────────────────────────────────────────
# Run this cell FIRST if you are on Google Colab or a fresh local environment.
# Skip it if you have already cloned the repo and installed requirements.
#
# !pip install -r requirements.txt
# !git clone https://github.com/YOUR_ORG/cinemastream.git
# import os; os.chdir("cinemastream")
# ─────────────────────────────────────────────────────────────────────────────
# Ensure the canonical dataset exists (deterministic; safe to re-run).
try:
    from cinemastream.scripts.generate_data import generate
    generate()
except ModuleNotFoundError:
    print("Run the clone/cd lines above first (Colab), then re-run this cell.")

## Chapters in this notebook

- [Chapter 52: Monitoring and Alerting](#chapter_52_monitoring_and_alerting)
- [Chapter 53: Data Observability](#chapter_53_data_observability)
- [Chapter 54: Data Contracts & API-First Data Sharing](#chapter_54_data_contracts_api_first_data_sharing)
- [Chapter 55: Incremental Processing & Change Data Capture (CDC)](#chapter_55_incremental_processing_change_data_capture_cdc)
- [Chapter 56: Cost Monitoring & FinOps for Data Pipelines](#chapter_56_cost_monitoring_finops_for_data_pipelines)
- [Chapter 57: Incident Response & Postmortems for Data Teams](#chapter_57_incident_response_postmortems_for_data_teams)
- [Chapter 58: Real-Time Architecture — Kafka, Kinesis, and Streaming Foundations](#chapter_58_real_time_architecture_kafka_kinesis_and_streaming_foundations)
- [Chapter 59: Data Mesh & Decentralized Ownership](#chapter_59_data_mesh_decentralized_ownership)

---

# Chapter 52: Monitoring and Alerting

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    P[Pipeline runs\non cron schedule] --> SIG[Signals collected\nliveness, row count, latency]
    SIG --> MON[Monitoring\nhistorical baseline]
    MON -->|within normal range| OK[Log success\nno action]
    MON -->|crosses threshold| ALERT[Alert fired]
    ALERT --> SL[Slack / email\npriya@cinemastream.com]
    ALERT --> INC[Incident created\non-call paged]
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install requests

### 2.1 Cron syntax — the universal scheduling language

```
┌───────────── minute (0-59)
│ ┌───────────── hour (0-23)
│ │ ┌───────────── day of month (1-31)
│ │ │ ┌───────────── month (1-12)
│ │ │ │ ┌───────────── day of week (0-6, Sunday=0)
│ │ │ │ │
* * * * *
```

In [ ]:
def explain_cron(expr: str) -> str:
    """Explain a 5-field cron expression in plain English (toy parser)."""
    fields = expr.split()
    if len(fields) != 5:
        raise ValueError(f"Expected 5 fields, got {len(fields)}: '{expr}'")

    minute, hour, dom, month, dow = fields

    parts = []

    # Time of day
    if minute != "*" and hour != "*":
        parts.append(f"at {hour.zfill(2)}:{minute.zfill(2)}")
    elif minute.startswith("*/"):
        parts.append(f"every {minute[2:]} minutes")
    else:
        parts.append(f"minute={minute}, hour={hour}")

    # Day of month
    if dom != "*":
        parts.append(f"on day {dom} of the month")

    # Month
    if month != "*":
        parts.append(f"in month {month}")

    # Day of week
    dow_names = {"0": "Sunday", "1": "Monday", "2": "Tuesday", "3": "Wednesday",
                  "4": "Thursday", "5": "Friday", "6": "Saturday"}
    if dow != "*":
        names = ", ".join(dow_names.get(d, d) for d in dow.split(","))
        parts.append(f"on {names}")

    return " ".join(parts)


examples = [
    "0 2 * * *",     # daily at 2am
    "*/15 * * * *",  # every 15 minutes
    "0 9 * * 1-5",   # weekday mornings
    "30 23 1 * *",   # 1st of every month
]

for expr in examples:
    print(f"{expr:15s} -> {explain_cron(expr)}")

```
0 2 * * *       -> at 02:00
*/15 * * * *    -> every 15 minutes
0 9 * * 1-5     -> at 09:00 on 1-5
30 23 1 * *     -> at 23:30 on day 1 of the month
```

### 2.2 A health check function — "did it run, and was it healthy?"

In [ ]:
from datetime import datetime, timedelta

def check_freshness(last_run: datetime, expected_interval_hours: float,
                     now: datetime = None) -> dict:
    """
    Compare a job's last successful run to how long ago it 'should' have run.

    Returns a dict with status: 'ok', 'late', or 'critical'.
    - ok:       within the expected interval
    - late:     up to 2x the expected interval overdue
    - critical: more than 2x overdue
    """
    if now is None:
        now = datetime.utcnow()

    elapsed = now - last_run
    expected = timedelta(hours=expected_interval_hours)

    if elapsed <= expected:
        status = "ok"
    elif elapsed <= expected * 2:
        status = "late"
    else:
        status = "critical"

    return {
        "last_run": last_run.isoformat(),
        "now": now.isoformat(),
        "elapsed_hours": round(elapsed.total_seconds() / 3600, 2),
        "expected_interval_hours": expected_interval_hours,
        "status": status,
    }


# Simulate three scenarios
now = datetime(2026, 6, 10, 9, 0, 0)

scenarios = {
    "healthy":  datetime(2026, 6, 10, 2, 5, 0),   # ran 2h ago, expected every 24h
    "late":     datetime(2026, 6, 8, 2, 0, 0),    # ran ~2 days ago
    "critical": datetime(2026, 6, 5, 2, 0, 0),    # ran 5 days ago
}

for label, last_run in scenarios.items():
    result = check_freshness(last_run, expected_interval_hours=24, now=now)
    print(f"{label:10s} -> {result}")

```
healthy    -> {'last_run': '2026-06-10T02:05:00', 'now': '2026-06-10T09:00:00', 'elapsed_hours': 6.92, 'expected_interval_hours': 24, 'status': 'ok'}
late       -> {'last_run': '2026-06-08T02:00:00', 'now': '2026-06-10T09:00:00', 'elapsed_hours': 55.0, 'expected_interval_hours': 24, 'status': 'late'}
critical   -> {'last_run': '2026-06-05T02:00:00', 'now': '2026-06-10T09:00:00', 'elapsed_hours': 127.0, 'expected_interval_hours': 24, 'status': 'critical'}
```

### 2.3 Volume anomaly detection — "did it process a normal amount?"

In [ ]:
import statistics

def check_row_count_anomaly(today_count: int, history: list[int],
                              z_threshold: float = 2.0) -> dict:
    """
    Compare today's row count to a rolling history using a z-score.

    z-score = (today - mean) / stdev
    A |z-score| above z_threshold is flagged as anomalous.
    """
    mean = statistics.mean(history)
    stdev = statistics.stdev(history) if len(history) > 1 else 0

    if stdev == 0:
        z_score = 0.0
    else:
        z_score = (today_count - mean) / stdev

    is_anomaly = abs(z_score) > z_threshold

    return {
        "today_count": today_count,
        "history_mean": round(mean, 1),
        "history_stdev": round(stdev, 1),
        "z_score": round(z_score, 2),
        "is_anomaly": is_anomaly,
    }


# 7 days of "normal" row counts, then test today's value
history = [38_200, 39_100, 37_800, 40_500, 38_900, 39_300, 38_600]

for today in [39_000, 41_200, 2_100]:
    result = check_row_count_anomaly(today, history)
    print(result)

```
{'today_count': 39000, 'history_mean': 38914.3, 'history_stdev': 870.7, 'z_score': 0.1, 'is_anomaly': False}
{'today_count': 41200, 'history_mean': 38914.3, 'history_stdev': 870.7, 'z_score': 2.63, 'is_anomaly': True}
{'today_count': 2100, 'history_mean': 38914.3, 'history_stdev': 870.7, 'z_score': -42.28, 'is_anomaly': True}
```

### 2.4 A Slack webhook alert function

In [ ]:
import json

def build_slack_alert(check_name: str, status: str, details: dict) -> dict:
    """
    Build a Slack webhook payload for a monitoring alert.
    Does NOT send it — returns the payload dict for inspection/testing.

    Severity emoji mapping keeps the message scannable in a busy channel.
    """
    emoji = {"ok": ":white_check_mark:", "late": ":warning:", "critical": ":rotating_light:"}
    color = {"ok": "#3D7370", "late": "#E07A3B", "critical": "#D62728"}

    detail_lines = "\n".join(f"  • {k}: {v}" for k, v in details.items())

    payload = {
        "text": f"{emoji.get(status, ':grey_question:')} *{check_name}* — status: `{status}`",
        "attachments": [
            {
                "color": color.get(status, "#888888"),
                "text": detail_lines,
            }
        ],
    }
    return payload


payload = build_slack_alert(
    check_name="watch_events_ingestion freshness",
    status="critical",
    details={"last_run": "2026-06-05T02:00:00Z", "elapsed_hours": 127.0, "expected_hours": 24},
)
print(json.dumps(payload, indent=2))

```
{
  "text": ":rotating_light: *watch_events_ingestion freshness* \u2014 status: `critical`",
  "attachments": [
    {
      "color": "#D62728",
      "text": "  \u2022 last_run: 2026-06-05T02:00:00Z\n  \u2022 elapsed_hours: 127.0\n  \u2022 expected_hours: 24"
    }
  ]
}
```

### 2.5 On-call basics — what happens after the alert fires

In [ ]:
def classify_severity(check_result: dict) -> str:
    """
    Map a check's status to a severity level using simple rules.
    Real systems often combine multiple signals (status + business hours +
    which table is affected) to decide severity.
    """
    status = check_result.get("status") or ("anomaly" if check_result.get("is_anomaly") else "ok")

    if status == "critical" or status == "anomaly" and check_result.get("z_score", 0) < -10:
        return "P1"
    elif status in ("late", "anomaly"):
        return "P2"
    else:
        return "P3"


checks = [
    {"status": "ok"},
    {"status": "late"},
    {"status": "critical"},
    {"is_anomaly": True, "z_score": -41.97},   # the 2,100-row scenario from 2.3
    {"is_anomaly": True, "z_score": 2.61},     # the 41,200-row scenario from 2.3
]

for c in checks:
    print(f"{c} -> {classify_severity(c)}")

```
{'status': 'ok'} -> P3
{'status': 'late'} -> P2
{'status': 'critical'} -> P1
{'is_anomaly': True, 'z_score': -41.97} -> P1
{'is_anomaly': True, 'z_score': 2.61} -> P2
```

## 3. CinemaStream in Practice

In [ ]:
import statistics
from datetime import datetime, timedelta

def run_pipeline_health_check(loaded_rows: int, history_counts: list[int],
                                last_run: datetime, expected_interval_hours: float = 25,
                                now: datetime = None) -> dict:
    """
    CinemaStream pipeline health check — combines freshness + volume checks
    into a single report with an overall severity.
    """
    if now is None:
        now = datetime.utcnow()

    # Freshness
    elapsed = now - last_run
    expected = timedelta(hours=expected_interval_hours)
    if elapsed <= expected:
        freshness_status = "ok"
    elif elapsed <= expected * 2:
        freshness_status = "late"
    else:
        freshness_status = "critical"

    # Volume (z-score)
    mean = statistics.mean(history_counts)
    stdev = statistics.stdev(history_counts) if len(history_counts) > 1 else 0
    z_score = (loaded_rows - mean) / stdev if stdev else 0.0
    volume_anomaly = abs(z_score) > 2.0

    # Overall severity — worst of the two checks wins
    if freshness_status == "critical" or (volume_anomaly and z_score < -10):
        severity = "P1"
    elif freshness_status == "late" or volume_anomaly:
        severity = "P2"
    else:
        severity = "P3"

    return {
        "loaded_rows": loaded_rows,
        "history_mean": round(mean, 1),
        "z_score": round(z_score, 2),
        "volume_anomaly": volume_anomaly,
        "freshness_status": freshness_status,
        "elapsed_hours": round(elapsed.total_seconds() / 3600, 2),
        "severity": severity,
    }


# Friday's incident: 0 rows loaded, last run was 8 hours ago (on schedule)
friday_history = [381, 392, 374, 405, 388, 397, 390]  # CinemaStream's daily watch_events count
result = run_pipeline_health_check(
    loaded_rows=0,
    history_counts=friday_history,
    last_run=datetime(2026, 6, 5, 2, 0, 0),
    now=datetime(2026, 6, 5, 10, 0, 0),
)
print(result)

```
{'loaded_rows': 0, 'history_mean': 389.6, 'z_score': -3.18, 'volume_anomaly': True, 'freshness_status': 'ok', 'elapsed_hours': 8.0, 'severity': 'P2'}
```

In [ ]:
def monitor_pipeline_health(**context) -> None:
    """
    Task: monitor (NEW — Chapter 52)
    Runs after load. Checks freshness + volume against the last 7 days
    of loaded_rows counts (read from the warehouse), classifies severity,
    and sends a Slack alert if severity is P1 or P2.

    On P1, this task raises — which fails the DAG run and triggers
    Airflow's email_on_failure (already configured in DEFAULT_ARGS).
    """
    import os
    import sqlite3
    import requests
    from datetime import datetime

    ti = context["ti"]
    loaded_rows = ti.xcom_pull(task_ids="load", key="loaded_rows") or 0
    logical_date = context["logical_date"]

    # Pull last 7 days of loaded_rows from the warehouse for the history baseline
    warehouse_db = Path(__file__).parent.parent / "data" / "warehouse.db"
    conn = sqlite3.connect(str(warehouse_db))
    history_df = pd.read_sql(
        """
        SELECT DATE(loaded_at) AS load_date, COUNT(*) AS row_count
        FROM watch_events_daily
        GROUP BY DATE(loaded_at)
        ORDER BY load_date DESC
        LIMIT 8
        """,
        conn,
    )
    conn.close()

    # Exclude today's just-loaded rows from the history baseline
    history_counts = history_df["row_count"].tolist()[1:8] or [loaded_rows]

    health = run_pipeline_health_check(
        loaded_rows=loaded_rows,
        history_counts=history_counts,
        last_run=logical_date.replace(tzinfo=None),
        now=datetime.utcnow(),
    )

    logger.info("Health check: %s", health)
    ti.xcom_push(key="health_check", value=health)

    if health["severity"] in ("P1", "P2"):
        webhook_url = os.environ.get("SLACK_WEBHOOK_URL")
        payload = build_slack_alert(
            check_name="watch_events_ingestion daily health check",
            status="critical" if health["severity"] == "P1" else "late",
            details=health,
        )
        if webhook_url:
            requests.post(webhook_url, json=payload, timeout=10)
        else:
            logger.warning("SLACK_WEBHOOK_URL not set — alert NOT sent: %s", payload)

    if health["severity"] == "P1":
        raise ValueError(f"Pipeline health check FAILED (P1): {health}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
result = check_freshness(
    last_run=datetime(2026, 6, 10, 0, 0, 0),
    expected_interval_hours=6,
    now=datetime(2026, 6, 10, 8, 0, 0),  # 8 hours later
)
print(result["status"])

In [ ]:
import statistics

baseline = [395, 401, 388, 22, 410]   # first 5 days — note: includes an outlier (22)!
recent   = [392, 0]                    # last 2 days to evaluate

mean = statistics.mean(baseline)
stdev = statistics.stdev(baseline)

alert_days = 0
for day_value in recent:
    z = (day_value - mean) / stdev
    is_anomaly = abs(z) > 2.0
    severity = "P1" if (is_anomaly and z < -10) else ("P2" if is_anomaly else "P3")
    print(f"value={day_value}, z={z:.2f}, severity={severity}")
    if severity in ("P1", "P2"):
        alert_days += 1

print(f"\nDays that would alert: {alert_days}")

```
value=392, z=0.41, severity=P3
value=0, z=-1.92, severity=P3

Days that would alert: 0
```

---

# Chapter 53: Data Observability

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    DATA[Dataset arrives] --> PROF[Profile today's data\nschema + distributions]
    PROF --> DIFF[Compare to baseline\nyesterday's profile]
    DIFF -->|No change| OK[Observability check passed]
    DIFF -->|Schema drift| SD[Alert: column added\nor removed or retyped]
    DIFF -->|Volume anomaly| VA[Alert: row count\noutside normal range]
    DIFF -->|Null rate shift| NR[Alert: column X\n1% → 15% null]
    DIFF -->|Distribution shift| DS[Alert: device values\ninclude new category]
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

### 2.1 Schema fingerprinting and drift detection

In [ ]:
import pandas as pd

def schema_fingerprint(df: pd.DataFrame) -> dict:
    """Capture column names, order, and dtypes as a comparable snapshot."""
    return {col: str(dtype) for col, dtype in df.dtypes.items()}


def detect_schema_drift(old_schema: dict, new_schema: dict) -> dict:
    """
    Compare two schema fingerprints. Returns a structured diff:
    columns added, removed, or changed type.
    """
    old_cols = set(old_schema.keys())
    new_cols = set(new_schema.keys())

    added = new_cols - old_cols
    removed = old_cols - new_cols
    common = old_cols & new_cols

    type_changed = {
        col: (old_schema[col], new_schema[col])
        for col in common
        if old_schema[col] != new_schema[col]
    }

    return {
        "added_columns": sorted(added),
        "removed_columns": sorted(removed),
        "type_changed": type_changed,
        "has_drift": bool(added or removed or type_changed),
    }


# Yesterday's schema
yesterday = pd.DataFrame({
    "order_id": [1, 2],
    "city": ["Delhi", "Mumbai"],
    "amount": [450.0, 700.0],
})

# Today: a new column appeared, and amount became a string (!)
today = pd.DataFrame({
    "order_id": [3, 4],
    "city": ["Pune", "Chennai"],
    "amount": ["650", "800"],          # type changed: float -> object
    "promo_code": ["WELCOME10", None], # new column
})

old_schema = schema_fingerprint(yesterday)
new_schema = schema_fingerprint(today)

print(detect_schema_drift(old_schema, new_schema))

```
{'added_columns': ['promo_code'], 'removed_columns': [], 'type_changed': {'amount': ('float64', 'object')}, 'has_drift': True}
```

### 2.2 Null-rate drift

In [ ]:
def null_rate_report(df: pd.DataFrame) -> dict:
    """Compute the fraction of NULL/NaN values per column."""
    return (df.isnull().sum() / len(df)).round(4).to_dict()


def detect_null_rate_drift(baseline: dict, current: dict, threshold: float = 0.05) -> dict:
    """
    Compare null rates between a baseline and current snapshot.
    Flags columns where the null rate changed by more than `threshold`
    (as an absolute difference, e.g., 0.01 -> 0.16 is a 0.15 jump).
    """
    drifted = {}
    for col, current_rate in current.items():
        baseline_rate = baseline.get(col, 0.0)
        delta = round(current_rate - baseline_rate, 4)
        if abs(delta) > threshold:
            drifted[col] = {"baseline": baseline_rate, "current": current_rate, "delta": delta}
    return drifted


# Baseline (yesterday): watch_minutes almost never NULL
baseline_df = pd.DataFrame({
    "event_id": [1, 2, 3, 4],
    "watch_minutes": [45, 90, 12, 60],
    "device": ["Mobile", "TV", "Web", "Tablet"],
})

# Today: watch_minutes is mostly NULL — something broke upstream
today_df = pd.DataFrame({
    "event_id": [5, 6, 7, 8],
    "watch_minutes": [None, None, None, 30],
    "device": ["Mobile", "TV", "Web", "Tablet"],
})

baseline_nulls = null_rate_report(baseline_df)
today_nulls = null_rate_report(today_df)

print("baseline:", baseline_nulls)
print("today:   ", today_nulls)
print("drift:   ", detect_null_rate_drift(baseline_nulls, today_nulls))

```
baseline: {'event_id': 0.0, 'watch_minutes': 0.0, 'device': 0.0}
today:    {'event_id': 0.0, 'watch_minutes': 0.75, 'device': 0.0}
drift:    {'watch_minutes': {'baseline': 0.0, 'current': 0.75, 'delta': 0.75}}
```

### 2.3 Distribution shift — categorical and numeric

In [ ]:
def detect_categorical_drift(baseline: pd.Series, current: pd.Series) -> dict:
    """Detect new or disappeared categories in a categorical column.
    .tolist() converts numpy scalar types (e.g., np.bool_, np.int64) found in
    .unique() back to plain Python types for clean printing/JSON output."""
    baseline_values = set(pd.Series(baseline.dropna().unique()).tolist())
    current_values = set(pd.Series(current.dropna().unique()).tolist())

    return {
        "new_values": sorted(current_values - baseline_values, key=str),
        "missing_values": sorted(baseline_values - current_values, key=str),
    }


def detect_numeric_range_shift(baseline: pd.Series, current: pd.Series) -> dict:
    """Compare min/max/mean between a baseline and current numeric column.
    Wraps numpy scalars with int()/float()/bool() so the result is JSON-safe."""
    return {
        "baseline": {"min": int(baseline.min()), "max": int(baseline.max()), "mean": round(float(baseline.mean()), 2)},
        "current":  {"min": int(current.min()),  "max": int(current.max()),  "mean": round(float(current.mean()), 2)},
        "max_increased": bool(current.max() > baseline.max()),
        "min_decreased": bool(current.min() < baseline.min()),
    }


baseline_devices = pd.Series(["Mobile", "TV", "Web", "Tablet", "Mobile", "TV"])
current_devices = pd.Series(["Mobile", "TV", "Smart_TV", "Mobile", "Smart_TV"])  # new device type!

print("categorical drift:", detect_categorical_drift(baseline_devices, current_devices))

baseline_minutes = pd.Series([10, 25, 45, 90, 60, 30])
current_minutes = pd.Series([10, 25, 45, 90, 60, 480])  # 480 minutes = 8 hours, way out of range

print("numeric drift:   ", detect_numeric_range_shift(baseline_minutes, current_minutes))

```
categorical drift: {'new_values': ['Smart_TV'], 'missing_values': ['Tablet', 'Web']}
numeric drift:     {'baseline': {'min': 10, 'max': 90, 'mean': 43.33}, 'current': {'min': 10, 'max': 480, 'mean': 118.33}, 'max_increased': True, 'min_decreased': False}
```

### 2.4 Putting it together — an observability report

In [ ]:
def observability_report(baseline_df: pd.DataFrame, current_df: pd.DataFrame,
                          numeric_cols: list[str], categorical_cols: list[str]) -> dict:
    """
    Run the full observability suite: schema, null-rate, and distribution checks.
    Returns a single report dict suitable for logging or alerting (Chapter 52).
    """
    report = {
        "schema": detect_schema_drift(schema_fingerprint(baseline_df), schema_fingerprint(current_df)),
        "null_rate": detect_null_rate_drift(null_rate_report(baseline_df), null_rate_report(current_df)),
        "categorical": {},
        "numeric": {},
    }

    for col in categorical_cols:
        if col in baseline_df.columns and col in current_df.columns:
            report["categorical"][col] = detect_categorical_drift(baseline_df[col], current_df[col])

    for col in numeric_cols:
        if col in baseline_df.columns and col in current_df.columns:
            report["numeric"][col] = detect_numeric_range_shift(baseline_df[col], current_df[col])

    # A simple "anything to investigate?" flag
    report["any_drift"] = (
        report["schema"]["has_drift"]
        or bool(report["null_rate"])
        or any(v["new_values"] or v["missing_values"] for v in report["categorical"].values())
        or any(v["max_increased"] or v["min_decreased"] for v in report["numeric"].values())
    )
    return report


import json
report = observability_report(
    baseline_df=pd.DataFrame({
        "event_id": [1, 2, 3], "watch_minutes": [45, 90, 60], "device": ["Mobile", "TV", "Web"],
    }),
    current_df=pd.DataFrame({
        "event_id": [4, 5, 6], "watch_minutes": [None, None, 60], "device": ["Mobile", "Smart_TV", "Web"],
    }),
    numeric_cols=["watch_minutes"],
    categorical_cols=["device"],
)
print(json.dumps(report, indent=2, default=str))

```
{
  "schema": {
    "added_columns": [],
    "removed_columns": [],
    "type_changed": {
      "watch_minutes": [
        "int64",
        "float64"
      ]
    },
    "has_drift": true
  },
  "null_rate": {
    "watch_minutes": {
      "baseline": 0.0,
      "current": 0.6667,
      "delta": 0.6667
    }
  },
  "categorical": {
    "device": {
      "new_values": [
        "Smart_TV"
      ],
      "missing_values": [
        "TV"
      ]
    }
  },
  "numeric": {
    "watch_minutes": {
      "baseline": {
        "min": 45,
        "max": 90,
        "mean": 65.0
      },
      "current": {
        "min": 60,
        "max": 60,
        "mean": 60.0
      },
      "max_increased": false,
      "min_decreased": false
    }
  },
  "any_drift": true
}
```

## 3. CinemaStream in Practice

In [ ]:
import pandas as pd

# Simulating two days of CinemaStream watch_events for country='PH'
# (in production this would be two SELECT queries against the read replica)

monday_ph = pd.DataFrame({
    "event_id": [5001, 5002, 5003, 5004, 5005],
    "user_id": [201, 202, 203, 204, 205],
    "movie_id": [101, 102, 103, 101, 102],
    "watch_minutes": [120, 45, 96, 60, 38],
    "completed": [True, False, True, False, False],
    "device": ["TV", "Mobile", "Web", "TV", "Mobile"],
    "country": ["PH"] * 5,
})

tuesday_ph = pd.DataFrame({
    "event_id": [5101, 5102, 5103, 5104, 5105],
    "user_id": [206, 207, 208, 209, 210],
    "movie_id": [101, 102, 103, 101, 102],
    "watch_minutes": [9, 8, 11, 7, 10],   # <- collapsed from ~70 avg to ~9
    "completed": [False, False, False, False, False],
    "device": ["TV", "Mobile", "Web", "TV", "Mobile"],
    "country": ["PH"] * 5,
})

report = observability_report(
    baseline_df=monday_ph,
    current_df=tuesday_ph,
    numeric_cols=["watch_minutes"],
    categorical_cols=["device", "completed"],
)

print("Schema drift:    ", report["schema"]["has_drift"])
print("Null-rate drift: ", report["null_rate"])
print("watch_minutes:   ", report["numeric"]["watch_minutes"])
print("completed drift: ", report["categorical"]["completed"])
print("any_drift:       ", report["any_drift"])

```
Schema drift:     False
Null-rate drift:  {}
watch_minutes:    {'baseline': {'min': 38, 'max': 120, 'mean': 71.8}, 'current': {'min': 7, 'max': 11, 'mean': 9.0}, 'max_increased': False, 'min_decreased': True}
completed drift:  {'new_values': [], 'missing_values': [True]}
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
day1 = pd.Series(["active", "active", "inactive", "active", "pending"])
day2 = pd.Series(["active", "active", "inactive", "cancelled", "cancelled"])

In [ ]:
print(detect_categorical_drift(day1, day2))

```
{'new_values': ['cancelled'], 'missing_values': ['pending']}
```

In [ ]:
last_month_priority = pd.Series(["Low", "Medium", "High", "Critical", "Low", "Medium"])
this_month_priority = pd.Series(["Low", "Medium", "High", "Urgent", "Urgent", "Low"])

print(detect_categorical_drift(last_month_priority, this_month_priority))

```
{'new_values': ['Urgent'], 'missing_values': ['Critical']}
```

---

# Chapter 54: Data Contracts & API-First Data Sharing

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    PROD[Producer team\nowns watch_events] --> CONTRACT[Data Contract v1\nschema + SLAs + semantics]
    CONTRACT --> CONS1[Consumer: dbt models]
    CONTRACT --> CONS2[Consumer: ML pipeline]
    CONTRACT --> CONS3[Consumer: dashboard]
    PROD -->|additive change| CV[Contract v2\nnew optional column]
    PROD -->|breaking change| BREAK[Rename column\nversioned as v2]
    BREAK --> NOTIFY[Notify all consumers\nbefore shipping]
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

### 2.1 Defining a contract as a schema specification

In [ ]:
# A data contract for a toy "orders" dataset, version 1
orders_contract_v1 = {
    "dataset": "orders",
    "version": 1,
    "fields": {
        "order_id":   {"type": "int",   "nullable": False, "unique": True},
        "city":       {"type": "str",   "nullable": False, "allowed_values": ["Delhi", "Mumbai", "Pune"]},
        "amount":     {"type": "float", "nullable": False, "min": 0},
        "status":     {"type": "str",   "nullable": False, "allowed_values": ["pending", "shipped", "delivered"]},
    },
}

print(f"Contract: {orders_contract_v1['dataset']} v{orders_contract_v1['version']}")
print(f"Required fields: {list(orders_contract_v1['fields'].keys())}")

```
Contract: orders v1
Required fields: ['order_id', 'city', 'amount', 'status']
```

### 2.2 Validating data against a contract

In [ ]:
import pandas as pd

PYTHON_TYPE_MAP = {"int": "int64", "float": "float64", "str": "object", "bool": "bool"}

def validate_against_contract(df: pd.DataFrame, contract: dict) -> list[str]:
    """Validate a DataFrame against a data contract spec. Returns a list of
    violation messages — empty list means the data conforms to the contract."""
    violations = []
    expected_fields = contract["fields"]

    # Missing required fields
    for field_name, spec in expected_fields.items():
        if field_name not in df.columns:
            violations.append(f"MISSING FIELD: '{field_name}' is in the contract but not in the data")
            continue

        col = df[field_name]

        # Type check (allow int -> float upcasting, which pandas does for NULL-able ints)
        expected_dtype = PYTHON_TYPE_MAP[spec["type"]]
        actual_dtype = str(col.dtype)
        if expected_dtype == "int64" and actual_dtype == "float64":
            pass  # tolerated: int column became float due to NULLs (Chapter 53)
        elif actual_dtype != expected_dtype:
            violations.append(f"TYPE MISMATCH: '{field_name}' expected {expected_dtype}, got {actual_dtype}")

        # Nullability
        if not spec.get("nullable", True) and col.isnull().any():
            violations.append(f"NULL VIOLATION: '{field_name}' has {col.isnull().sum()} NULLs but is non-nullable")

        # Allowed values
        if "allowed_values" in spec:
            bad_values = set(col.dropna().unique()) - set(spec["allowed_values"])
            if bad_values:
                violations.append(f"INVALID VALUES: '{field_name}' has unexpected values {sorted(bad_values)}")

        # Min value
        if "min" in spec and pd.api.types.is_numeric_dtype(col):
            below_min = col[col < spec["min"]]
            if len(below_min) > 0:
                violations.append(f"RANGE VIOLATION: '{field_name}' has {len(below_min)} values below {spec['min']}")

        # Uniqueness
        if spec.get("unique") and col.duplicated().any():
            violations.append(f"UNIQUENESS VIOLATION: '{field_name}' has {col.duplicated().sum()} duplicates")

    # Unexpected extra fields (informational, not necessarily a failure)
    extra_fields = set(df.columns) - set(expected_fields.keys())
    if extra_fields:
        violations.append(f"INFO: extra fields not in contract (additive, usually OK): {sorted(extra_fields)}")

    return violations


# A clean dataset that conforms to orders_contract_v1
good_orders = pd.DataFrame({
    "order_id": [1, 2, 3],
    "city": ["Delhi", "Mumbai", "Pune"],
    "amount": [450.0, 700.0, 320.0],
    "status": ["pending", "shipped", "delivered"],
})

print("Clean data:", validate_against_contract(good_orders, orders_contract_v1))

# A dataset with three problems
bad_orders = pd.DataFrame({
    "order_id": [1, 1, 3],                            # duplicate order_id
    "city": ["Delhi", "Bangalore", "Pune"],           # 'Bangalore' not in allowed_values
    "amount": [450.0, -50.0, 320.0],                  # negative amount
    "status": ["pending", "shipped", "delivered"],
})

print("Bad data:  ", validate_against_contract(bad_orders, orders_contract_v1))

```
Clean data: []
Bad data:   ["UNIQUENESS VIOLATION: 'order_id' has 1 duplicates", "INVALID VALUES: 'city' has unexpected values ['Bangalore']", "RANGE VIOLATION: 'amount' has 1 values below 0"]
```

### 2.3 Versioning and breaking vs non-breaking changes

In [ ]:
def classify_contract_change(old_contract: dict, new_contract: dict) -> dict:
    """
    Compare two versions of a contract and classify the change as
    'breaking', 'non-breaking', or 'no-change'.

    Breaking changes: removed fields, type changes, a field becoming
    non-nullable when it previously allowed nulls, or removing previously
    allowed_values.

    Non-breaking changes: new fields, a field becoming MORE permissive
    (nullable=True when it was False, or new allowed_values added).
    """
    old_fields = old_contract["fields"]
    new_fields = new_contract["fields"]

    breaking = []
    non_breaking = []

    # Removed fields
    for field_name in old_fields:
        if field_name not in new_fields:
            breaking.append(f"REMOVED: '{field_name}'")

    # Added fields
    for field_name in new_fields:
        if field_name not in old_fields:
            non_breaking.append(f"ADDED: '{field_name}'")

    # Changed fields
    for field_name in set(old_fields) & set(new_fields):
        old_spec, new_spec = old_fields[field_name], new_fields[field_name]

        if old_spec["type"] != new_spec["type"]:
            breaking.append(f"TYPE CHANGED: '{field_name}' {old_spec['type']} -> {new_spec['type']}")

        old_nullable = old_spec.get("nullable", True)
        new_nullable = new_spec.get("nullable", True)
        if old_nullable and not new_nullable:
            breaking.append(f"NOW NON-NULLABLE: '{field_name}'")
        elif not old_nullable and new_nullable:
            non_breaking.append(f"NOW NULLABLE: '{field_name}' (more permissive)")

        if "allowed_values" in old_spec and "allowed_values" in new_spec:
            removed_values = set(old_spec["allowed_values"]) - set(new_spec["allowed_values"])
            added_values = set(new_spec["allowed_values"]) - set(old_spec["allowed_values"])
            if removed_values:
                breaking.append(f"REMOVED ALLOWED VALUES: '{field_name}' lost {sorted(removed_values)}")
            if added_values:
                non_breaking.append(f"ADDED ALLOWED VALUES: '{field_name}' gained {sorted(added_values)}")

    if not breaking and not non_breaking:
        change_type = "no-change"
    elif breaking:
        change_type = "breaking"
    else:
        change_type = "non-breaking"

    return {"breaking": breaking, "non_breaking": non_breaking, "change_type": change_type}


# v2: 'amount' is renamed conceptually to mean "in cents" -> type changes int,
# a new 'currency' field is added, and 'cancelled' is added to status values.
orders_contract_v2 = {
    "dataset": "orders",
    "version": 2,
    "fields": {
        "order_id":   {"type": "int",   "nullable": False, "unique": True},
        "city":       {"type": "str",   "nullable": False, "allowed_values": ["Delhi", "Mumbai", "Pune"]},
        "amount":     {"type": "int",   "nullable": False, "min": 0},  # float -> int (now cents)
        "status":     {"type": "str",   "nullable": False, "allowed_values": ["pending", "shipped", "delivered", "cancelled"]},
        "currency":   {"type": "str",   "nullable": False, "allowed_values": ["INR"]},
    },
}

print(classify_contract_change(orders_contract_v1, orders_contract_v2))

```
{'breaking': ["TYPE CHANGED: 'amount' float -> int"], 'non_breaking': ["ADDED: 'currency'", "ADDED ALLOWED VALUES: 'status' gained ['cancelled']"], 'change_type': 'breaking'}
```

## 3. CinemaStream in Practice

In [ ]:
watch_events_contract_v1 = {
    "dataset": "watch_events",
    "version": 1,
    "fields": {
        "event_id":      {"type": "int",   "nullable": False, "unique": True},
        "user_id":       {"type": "int",   "nullable": False},
        "movie_id":      {"type": "int",   "nullable": False},
        "watch_minutes": {"type": "int",   "nullable": False, "min": 1},
        "completed":     {"type": "bool",  "nullable": False, "allowed_values": [True, False]},
        "device":        {"type": "str",   "nullable": False, "allowed_values": ["Mobile", "TV", "Web", "Tablet"]},
        "country":       {"type": "str",   "nullable": False, "allowed_values": ["SG", "MY", "ID", "PH", "TH", "VN", "IN"]},
    },
}

# Carlos's proposed v2: completed becomes a string status; subtitle_lang is added
watch_events_contract_v2_proposed = {
    "dataset": "watch_events",
    "version": 2,
    "fields": {
        "event_id":      {"type": "int",   "nullable": False, "unique": True},
        "user_id":       {"type": "int",   "nullable": False},
        "movie_id":      {"type": "int",   "nullable": False},
        "watch_minutes": {"type": "int",   "nullable": False, "min": 1},
        "completed":     {"type": "str",   "nullable": False, "allowed_values": ["not_started", "in_progress", "completed"]},
        "device":        {"type": "str",   "nullable": False, "allowed_values": ["Mobile", "TV", "Web", "Tablet"]},
        "country":       {"type": "str",   "nullable": False, "allowed_values": ["SG", "MY", "ID", "PH", "TH", "VN", "IN"]},
        "subtitle_lang": {"type": "str",   "nullable": True,  "allowed_values": ["en", "ms", "id", "tl", "th", "vi", "hi", "ta", None]},
    },
}

result = classify_contract_change(watch_events_contract_v1, watch_events_contract_v2_proposed)
print(f"Change type: {result['change_type']}")
print("Breaking changes:")
for b in result["breaking"]:
    print(f"  - {b}")
print("Non-breaking changes:")
for nb in result["non_breaking"]:
    print(f"  - {nb}")

```
Change type: breaking
Breaking changes:
  - TYPE CHANGED: 'completed' bool -> str
  - REMOVED ALLOWED VALUES: 'completed' lost [False, True]
Non-breaking changes:
  - ADDED: 'subtitle_lang'
  - ADDED ALLOWED VALUES: 'completed' gained ['completed', 'in_progress', 'not_started']
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
contract_a = {"dataset": "users", "version": 1, "fields": {
    "user_id": {"type": "int", "nullable": False, "unique": True},
    "email":   {"type": "str", "nullable": False},
}}

contract_b = {"dataset": "users", "version": 2, "fields": {
    "user_id": {"type": "int", "nullable": False, "unique": True},
    "email":   {"type": "str", "nullable": True},   # now nullable
}}

In [ ]:
print(classify_contract_change(contract_a, contract_b))

```
{'breaking': [], 'non_breaking': ["NOW NULLABLE: 'email' (more permissive)"], 'change_type': 'non-breaking'}
```

In [ ]:
subs_contract_v1 = {"dataset": "subscriptions", "version": 1, "fields": {
    "amount_local": {"type": "float", "nullable": False, "min": 0},
    "currency":     {"type": "str",   "nullable": True},
    "status":       {"type": "str",   "nullable": False, "allowed_values": ["active", "cancelled"]},
}}

subs_contract_v2 = {"dataset": "subscriptions", "version": 2, "fields": {
    "amount_local": {"type": "float", "nullable": False, "min": 0},
    "currency":     {"type": "str",   "nullable": False},  # now required
    "status":       {"type": "str",   "nullable": False, "allowed_values": ["active", "cancelled", "cancelled_refunded"]},
}}

result = classify_contract_change(subs_contract_v1, subs_contract_v2)
print(result)

```
{'breaking': ["NOW NON-NULLABLE: 'currency'"], 'non_breaking': ["ADDED ALLOWED VALUES: 'status' gained ['cancelled_refunded']"], 'change_type': 'breaking'}
```

---

# Chapter 55: Incremental Processing & Change Data Capture (CDC)

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    SRC[Source DB\nwatch_events] --> FL[Full load\nre-read all rows every run]
    SRC --> INC[Incremental load\nupdated_at > high-water mark]
    SRC --> CDC[CDC\nread transaction log]
    FL -->|simple, expensive| WH[Warehouse]
    INC -->|cheap, misses DELETEs| WH
    CDC -->|INSERT + UPDATE + DELETE| WH
    LATE[Late-arriving events\nmobile offline sync] --> INC
    LATE --> CDC
</div>
"""))

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

### 2.1 Full load vs incremental load — the basic mechanics

In [ ]:
import pandas as pd

# A toy "source table" — simulates rows with an updated_at timestamp
source_table = pd.DataFrame({
    "order_id":   [1, 2, 3, 4, 5],
    "status":     ["delivered", "delivered", "shipped", "pending", "pending"],
    "updated_at": pd.to_datetime([
        "2026-06-01 10:00", "2026-06-02 14:30", "2026-06-03 09:15",
        "2026-06-04 16:45", "2026-06-05 11:20",
    ]),
})

def full_load(source: pd.DataFrame) -> pd.DataFrame:
    """Re-process the entire source table. Simple, but O(n) every run."""
    return source.copy()

def incremental_load(source: pd.DataFrame, high_water_mark: pd.Timestamp) -> pd.DataFrame:
    """Process only rows updated after the high-water mark."""
    return source[source["updated_at"] > high_water_mark].copy()


# A full load on day 5 processes all 5 rows
print("Full load (5 rows expected):")
print(full_load(source_table))

# An incremental load using yesterday's high-water mark processes only new/changed rows
hwm = pd.Timestamp("2026-06-04 16:45")
print("\nIncremental load since 2026-06-04 16:45 (1 row expected):")
print(incremental_load(source_table, hwm))

```
Full load (5 rows expected):
   order_id     status         updated_at
0         1  delivered 2026-06-01 10:00:00
1         2  delivered 2026-06-02 14:30:00
2         3    shipped 2026-06-03 09:15:00
3         4    pending 2026-06-04 16:45:00
4         5    pending 2026-06-05 11:20:00

Incremental load since 2026-06-04 16:45 (1 row expected):
   order_id  status          updated_at
4         5  pending 2026-06-05 11:20:00
```

### 2.2 Tracking the high-water mark

In [ ]:
def run_incremental_pipeline(source: pd.DataFrame, last_high_water_mark: pd.Timestamp) -> dict:
    """
    Run one incremental load and return the new high-water mark to persist.
    The new high-water mark is the MAX updated_at among processed rows —
    NOT "now", because "now" could be ahead of data still being written.
    """
    new_rows = incremental_load(source, last_high_water_mark)

    if len(new_rows) == 0:
        return {"processed_rows": 0, "new_high_water_mark": last_high_water_mark}

    new_hwm = new_rows["updated_at"].max()
    return {"processed_rows": len(new_rows), "new_high_water_mark": new_hwm, "data": new_rows}


# Simulate three consecutive runs
hwm = pd.Timestamp("2026-06-01 00:00")  # initial backfill point

for run_num in range(1, 4):
    result = run_incremental_pipeline(source_table, hwm)
    print(f"Run {run_num}: processed {result['processed_rows']} rows, "
          f"new HWM = {result['new_high_water_mark']}")
    hwm = result["new_high_water_mark"]

```
Run 1: processed 5 rows, new HWM = 2026-06-05 11:20:00
Run 2: processed 0 rows, new HWM = 2026-06-05 11:20:00
Run 3: processed 0 rows, new HWM = 2026-06-05 11:20:00
```

### 2.3 Simulating CDC — insert, update, delete events

In [ ]:
def apply_cdc_events(target: pd.DataFrame, cdc_events: list[dict]) -> pd.DataFrame:
    """
    Apply a sequence of CDC events (insert/update/delete) to a target DataFrame.
    Each event: {"op": "insert"|"update"|"delete", "order_id": ..., "data": {...}}
    """
    target = target.set_index("order_id", drop=False)

    for event in cdc_events:
        op = event["op"]
        order_id = event["order_id"]

        if op == "insert":
            target.loc[order_id] = event["data"]
        elif op == "update":
            for col, val in event["data"].items():
                target.loc[order_id, col] = val
        elif op == "delete":
            target = target.drop(index=order_id, errors="ignore")
        else:
            raise ValueError(f"Unknown CDC operation: {op}")

    return target.reset_index(drop=True)


# Starting state
target_table = pd.DataFrame({
    "order_id": [1, 2, 3],
    "status": ["pending", "pending", "shipped"],
})

# A CDC log: order 1 gets shipped, order 4 is a new insert, order 2 is cancelled (deleted)
cdc_log = [
    {"op": "update", "order_id": 1, "data": {"status": "shipped"}},
    {"op": "insert", "order_id": 4, "data": {"order_id": 4, "status": "pending"}},
    {"op": "delete", "order_id": 2},
]

print("Before:")
print(target_table)
print("\nAfter applying CDC log:")
print(apply_cdc_events(target_table, cdc_log))

```
Before:
   order_id   status
0         1  pending
1         2  pending
2         3  shipped

After applying CDC log:
   order_id   status
0         1  shipped
1         3  shipped
2         4  pending
```

### 2.4 Late-arriving data and reconciliation

In [ ]:
from datetime import datetime, timedelta

def partition_by_event_date(events: pd.DataFrame, event_time_col: str, processing_date: str) -> dict:
    """
    Split events into 'on-time' (event happened on processing_date) and
    'late-arriving' (event's timestamp belongs to an EARLIER date than
    when it was actually loaded).
    """
    events = events.copy()
    events["event_date"] = events[event_time_col].dt.date
    proc_date = pd.Timestamp(processing_date).date()

    on_time = events[events["event_date"] == proc_date]
    late = events[events["event_date"] < proc_date]
    future = events[events["event_date"] > proc_date]

    return {"on_time": on_time, "late_arriving": late, "future_dated": future}


# Today's "load run" pulls events with a loaded_at timestamp of today,
# but their actual watch_started timestamps span two different days.
batch = pd.DataFrame({
    "event_id": [901, 902, 903, 904],
    "watch_started": pd.to_datetime([
        "2026-06-09 23:58",  # late-arriving — belongs to YESTERDAY's partition
        "2026-06-10 00:05",
        "2026-06-10 08:30",
        "2026-06-09 22:10",  # also late-arriving
    ]),
})

result = partition_by_event_date(batch, "watch_started", processing_date="2026-06-10")
print(f"On-time (belongs to today):     {len(result['on_time'])} rows")
print(f"Late-arriving (belongs to yesterday, needs reprocessing): {len(result['late_arriving'])} rows")
print(result["late_arriving"][["event_id", "watch_started"]])

```
On-time (belongs to today):     2 rows
Late-arriving (belongs to yesterday, needs reprocessing): 2 rows
   event_id       watch_started
0       901 2026-06-09 23:58:00
3       904 2026-06-09 22:10:00
```

## 3. CinemaStream in Practice

In [ ]:
import pandas as pd

# Simulating watch_events as they exist in the source RIGHT NOW (Carlos's
# read-replica), versus what the warehouse currently has loaded for
# 2026-06-09 (yesterday's "completed" partition).

source_now = pd.DataFrame({
    "event_id": [9001, 9002, 9003, 9004, 9005],
    "user_id":  [301, 302, 303, 304, 305],
    "watch_started": pd.to_datetime([
        "2026-06-09 14:00",  # loaded yesterday, on time
        "2026-06-09 19:30",  # loaded yesterday, on time
        "2026-06-09 23:50",  # PH user, app synced this morning -> LATE
        "2026-06-10 02:15",  # belongs to today
        "2026-06-09 21:05",  # VN user, app synced 18h late -> LATE
    ]),
    "country": ["SG", "MY", "PH", "ID", "VN"],
})

warehouse_loaded_2026_06_09 = pd.DataFrame({
    "event_id": [9001, 9002],   # only the two on-time events were loaded
    "user_id":  [301, 302],
    "watch_started": pd.to_datetime(["2026-06-09 14:00", "2026-06-09 19:30"]),
    "country": ["SG", "MY"],
})

# What SHOULD be in the 2026-06-09 partition, per the source right now
result = partition_by_event_date(source_now, "watch_started", processing_date="2026-06-09")
should_be_loaded = result["on_time"]

missing = should_be_loaded[~should_be_loaded["event_id"].isin(warehouse_loaded_2026_06_09["event_id"])]

print(f"Events that SHOULD be in the 2026-06-09 partition: {len(should_be_loaded)}")
print(f"Events currently loaded for 2026-06-09:           {len(warehouse_loaded_2026_06_09)}")
print(f"Missing (late-arriving, never reprocessed):       {len(missing)}")
print(missing[["event_id", "user_id", "watch_started", "country"]])

```
Events that SHOULD be in the 2026-06-09 partition: 4
Events currently loaded for 2026-06-09:           2
Missing (late-arriving, never reprocessed):       2
   event_id  user_id       watch_started country
2      9003      303 2026-06-09 23:50:00      PH
4      9005      305 2026-06-09 21:05:00      VN
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
cdc_log = [
    {"op": "insert", "order_id": 10, "data": {"order_id": 10, "status": "pending"}},
    {"op": "insert", "order_id": 11, "data": {"order_id": 11, "status": "pending"}},
    {"op": "update", "order_id": 10, "data": {"status": "shipped"}},
    {"op": "delete", "order_id": 11},
    {"op": "update", "order_id": 10, "data": {"status": "delivered"}},
]

In [ ]:
empty_target = pd.DataFrame(columns=["order_id", "status"])
print(apply_cdc_events(empty_target, cdc_log))

```
   order_id     status
0        10  delivered
```

In [ ]:
def upsert_simulation(existing_event_ids: set, incoming_event_ids: list) -> dict:
    """Simulate ON CONFLICT (event_id) DO NOTHING for a batch of incoming events."""
    inserted = [eid for eid in incoming_event_ids if eid not in existing_event_ids]
    skipped = [eid for eid in incoming_event_ids if eid in existing_event_ids]
    return {"inserted": inserted, "skipped_as_duplicate": skipped}

# ds=2026-06-09 run already loaded 9003
existing = {9001, 9002, 9003}

# ds=2026-06-10 run's lookback window re-extracts 9003, 9004, 9005
incoming = [9003, 9004, 9005]

print(upsert_simulation(existing, incoming))

```
{'inserted': [9004, 9005], 'skipped_as_duplicate': [9003]}
```

---

# Chapter 56: Cost Monitoring & FinOps for Data Pipelines

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

### 2.1 Estimating query cost from bytes scanned

In [ ]:
def estimate_query_cost(bytes_scanned: int, price_per_tb_usd: float = 5.0) -> dict:
    """
    Estimate the USD cost of a query given bytes scanned and a price per TB.
    $5/TB is BigQuery's on-demand pricing as of writing — a real,
    publicly published rate, used here for illustrative calculations.
    """
    tb_scanned = bytes_scanned / (1024 ** 4)
    cost_usd = tb_scanned * price_per_tb_usd
    return {
        "bytes_scanned": bytes_scanned,
        "gb_scanned": round(bytes_scanned / (1024 ** 3), 4),
        "tb_scanned": round(tb_scanned, 6),
        "estimated_cost_usd": round(cost_usd, 4),
    }


# A query that scans 50 GB
print(estimate_query_cost(50 * 1024**3))

# The same query, but the table is partitioned and the WHERE clause
# only touches 1 day's partition out of 365 -> ~50GB/365
print(estimate_query_cost(int(50 * 1024**3 / 365)))

```
{'bytes_scanned': 53687091200, 'gb_scanned': 50.0, 'tb_scanned': 0.048828, 'estimated_cost_usd': 0.2441}
{'bytes_scanned': 147087921, 'gb_scanned': 0.137, 'tb_scanned': 0.000134, 'estimated_cost_usd': 0.0007}
```

### 2.2 Cost attribution by tagging

In [ ]:
import pandas as pd

# A log of queries run over a week, tagged with team and project metadata
query_log = pd.DataFrame({
    "query_id":   ["q1", "q2", "q3", "q4", "q5", "q6"],
    "team":       ["analytics", "analytics", "ml", "ml", "analytics", "ml"],
    "pipeline":   ["daily_revenue", "daily_revenue", "churn_features", "churn_features", "ad_hoc", "churn_features"],
    "bytes_scanned": [
        2 * 1024**3,    # 2 GB
        2 * 1024**3,    # 2 GB (same pipeline, different day)
        500 * 1024**3,  # 500 GB — full scan!
        500 * 1024**3,
        15 * 1024**3,   # 15 GB ad-hoc query
        500 * 1024**3,
    ],
})

def attribute_costs(query_log: pd.DataFrame, price_per_tb_usd: float = 5.0) -> pd.DataFrame:
    """Compute per-query cost and aggregate by team and pipeline."""
    df = query_log.copy()
    df["cost_usd"] = df["bytes_scanned"] / (1024 ** 4) * price_per_tb_usd
    return df


costed = attribute_costs(query_log)
print(costed[["query_id", "team", "pipeline", "cost_usd"]].round(4))

print("\nCost by team:")
print(costed.groupby("team")["cost_usd"].sum().round(2))

print("\nCost by pipeline:")
print(costed.groupby("pipeline")["cost_usd"].sum().round(2).sort_values(ascending=False))

```
  query_id       team        pipeline  cost_usd
0       q1  analytics   daily_revenue    0.0098
1       q2  analytics   daily_revenue    0.0098
2       q3         ml  churn_features    2.4414
3       q4         ml  churn_features    2.4414
4       q5  analytics          ad_hoc    0.0732
5       q6         ml  churn_features    2.4414

Cost by team:
team
analytics    0.09
ml           7.32
Name: cost_usd, dtype: float64

Cost by pipeline:
pipeline
churn_features    7.32
ad_hoc            0.07
daily_revenue     0.02
Name: cost_usd, dtype: float64
```

### 2.3 Budget alerts

In [ ]:
def check_budget(actual_spend: float, budget: float, period: str = "monthly") -> dict:
    """
    Compare actual spend to a budget and classify status.
    Thresholds: 'ok' (<80%), 'warning' (80-100%), 'over_budget' (>100%).
    """
    pct_used = actual_spend / budget if budget > 0 else float("inf")

    if pct_used < 0.8:
        status = "ok"
    elif pct_used <= 1.0:
        status = "warning"
    else:
        status = "over_budget"

    return {
        "period": period,
        "actual_spend": round(float(actual_spend), 2),
        "budget": round(float(budget), 4),
        "pct_used": round(float(pct_used) * 100, 1),
        "status": status,
    }


# ml team's monthly budget for churn_features is $150
churn_monthly_spend = 7.3242 * 30  # extrapolate this week's run-rate to a month

print(check_budget(churn_monthly_spend, budget=150))

```
{'period': 'monthly', 'actual_spend': 219.73, 'budget': 150.0, 'pct_used': 146.5, 'status': 'over_budget'}
```

### 2.4 The optimization playbook — turning a cost problem into a fix

In [ ]:
def diagnose_high_cost_query(table_size_gb: float, bytes_scanned_gb: float,
                              is_partitioned: bool, columns_selected: int,
                              total_columns: int) -> list[str]:
    """
    A simple rules-based playbook: given basic facts about a query and its
    target table, suggest concrete optimizations (Chapter 50 techniques).
    """
    suggestions = []

    scan_ratio = bytes_scanned_gb / table_size_gb if table_size_gb > 0 else 0

    if not is_partitioned and scan_ratio > 0.5:
        suggestions.append(
            "Table is not partitioned and the query scans most of it. "
            "Add a partition column (e.g., DATE(watch_started)) and filter on it."
        )

    if columns_selected == total_columns:
        suggestions.append(
            f"Query selects all {total_columns} columns. SELECT only the "
            "columns you need — column-store warehouses (BigQuery, Snowflake) "
            "only bill for columns actually read."
        )

    if is_partitioned and scan_ratio > 0.3:
        suggestions.append(
            "Table is partitioned but the query still scans >30% of it. "
            "Check the WHERE clause is on the partition column directly "
            "(not wrapped in a function like DATE(timestamp_col) on an "
            "already-partitioned-by-date table, which can defeat pruning)."
        )

    if not suggestions:
        suggestions.append("No obvious issues — cost may simply reflect genuine data volume.")

    return suggestions


# churn_features query: 500GB table, scans all 500GB, not partitioned, selects all 40 columns
print(diagnose_high_cost_query(
    table_size_gb=500, bytes_scanned_gb=500,
    is_partitioned=False, columns_selected=40, total_columns=40,
))

```
['Table is not partitioned and the query scans most of it. Add a partition column (e.g., DATE(watch_started)) and filter on it.', 'Query selects all 40 columns. SELECT only the columns you need — column-store warehouses (BigQuery, Snowflake) only bill for columns actually read.']
```

## 3. CinemaStream in Practice

In [ ]:
import pandas as pd

last_month = pd.DataFrame({
    "pipeline": ["watch_events_ingestion", "mart_monthly_revenue", "churn_features", "ad_hoc_analytics"],
    "team":     ["data", "data", "ml", "analytics"],
    "bytes_scanned_gb": [12, 8, 45, 30],   # GB scanned, summed across the month
})

this_month = pd.DataFrame({
    "pipeline": ["watch_events_ingestion", "mart_monthly_revenue", "churn_features", "ad_hoc_analytics"],
    "team":     ["data", "data", "ml", "analytics"],
    "bytes_scanned_gb": [36, 8, 45, 30],   # watch_events_ingestion: 12 -> 36 (the 3x lookback!)
})

PRICE_PER_TB = 5.0

def compute_cost(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["cost_usd"] = df["bytes_scanned_gb"] / 1024 * PRICE_PER_TB
    return df

last_costed = compute_cost(last_month)
this_costed = compute_cost(this_month)

comparison = last_costed[["pipeline", "team", "cost_usd"]].rename(columns={"cost_usd": "last_month_usd"})
comparison["this_month_usd"] = this_costed["cost_usd"]
comparison["delta_usd"] = (comparison["this_month_usd"] - comparison["last_month_usd"]).round(4)
comparison["delta_pct"] = ((comparison["delta_usd"] / comparison["last_month_usd"]) * 100).round(1)

print(comparison.round(4).to_string(index=False))
print(f"\nTotal last month: ${comparison['last_month_usd'].sum():.4f}")
print(f"Total this month: ${comparison['this_month_usd'].sum():.4f}")
print(f"Total increase:   {((comparison['this_month_usd'].sum() / comparison['last_month_usd'].sum()) - 1) * 100:.1f}%")

```
              pipeline      team  last_month_usd  this_month_usd  delta_usd  delta_pct
watch_events_ingestion      data          0.0586          0.1758     0.1172      200.0
  mart_monthly_revenue      data          0.0391          0.0391     0.0000        0.0
        churn_features        ml          0.2197          0.2197     0.0000        0.0
      ad_hoc_analytics analytics          0.1465          0.1465     0.0000        0.0

Total last month: $0.4639
Total this month: $0.5811
Total increase:   25.3%
```

In [ ]:
# New monthly budget for watch_events_ingestion, set at ~130% of this month's
# actual to allow headroom for organic growth, but catch further step-changes
new_budget = round(comparison.loc[0, "this_month_usd"] * 1.3, 4)

print(check_budget(comparison.loc[0, "this_month_usd"], budget=new_budget))
print(f"Budget set at: ${new_budget}")

```
{'period': 'monthly', 'actual_spend': 0.18, 'budget': 0.2285, 'pct_used': 76.9, 'status': 'ok'}
Budget set at: $0.2285
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
# Partitioned: scans ~1/30th = ~6.67 GB
partitioned_cost = estimate_query_cost(int(200 / 30 * 1024**3))

# Unpartitioned: scans the full 200GB
unpartitioned_cost = estimate_query_cost(200 * 1024**3)

print("Partitioned:  ", partitioned_cost["estimated_cost_usd"])
print("Unpartitioned:", unpartitioned_cost["estimated_cost_usd"])
print("Difference:   ", round(unpartitioned_cost["estimated_cost_usd"] - partitioned_cost["estimated_cost_usd"], 4))

print(diagnose_high_cost_query(table_size_gb=200, bytes_scanned_gb=200,
                                is_partitioned=False, columns_selected=5, total_columns=20))

```
Partitioned:   0.0326
Unpartitioned: 0.9766
Difference:    0.944
```

```
['Table is not partitioned and the query scans most of it. Add a partition column (e.g., DATE(watch_started)) and filter on it.']
```

In [ ]:
current_monthly_cost = 0.1465  # from Section 3, comparison table

# 80% reduction in bytes scanned -> 80% reduction in cost (cost is linear in bytes)
new_monthly_cost = current_monthly_cost * (1 - 0.80)
monthly_savings = current_monthly_cost - new_monthly_cost
annual_savings = monthly_savings * 12

print(f"Current monthly cost: ${current_monthly_cost}")
print(f"New monthly cost:     ${new_monthly_cost:.4f}")
print(f"Monthly savings:      ${monthly_savings:.4f}")
print(f"Annual savings:       ${annual_savings:.4f}")

```
Current monthly cost: $0.1465
New monthly cost:     $0.0293
Monthly savings:      $0.1172
Annual savings:       $1.4064
```

---

# Chapter 57: Incident Response & Postmortems for Data Teams

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### 2.1 Modeling severity levels

In [ ]:
def classify_severity(
    users_affected_pct: float,
    revenue_impact: bool,
    workaround_exists: bool,
    data_loss: bool,
) -> dict:
    """
    Classify an incident's severity (SEV1 = most severe, SEV4 = least)
    based on a small set of objective signals.
    """
    if data_loss or (revenue_impact and not workaround_exists):
        severity = "SEV1"
        page = True
        response_sla_minutes = 15
    elif users_affected_pct >= 10 or revenue_impact:
        severity = "SEV2"
        page = True
        response_sla_minutes = 60
    elif users_affected_pct >= 1 or not workaround_exists:
        severity = "SEV3"
        page = False
        response_sla_minutes = 240  # 4 hours, business hours only
    else:
        severity = "SEV4"
        page = False
        response_sla_minutes = 1440  # next business day

    return {
        "severity": severity,
        "page_oncall": page,
        "response_sla_minutes": response_sla_minutes,
    }


# Scenario A: nightly pipeline fails, dashboard shows yesterday's data, refresh exists
print(classify_severity(users_affected_pct=100, revenue_impact=False,
                         workaround_exists=True, data_loss=False))

# Scenario B: a bad transform deletes 3 days of watch_events with no backup
print(classify_severity(users_affected_pct=100, revenue_impact=True,
                         workaround_exists=False, data_loss=True))

# Scenario C: one country's completion-rate metric is off by 2%, no one downstream affected yet
print(classify_severity(users_affected_pct=14, revenue_impact=False,
                         workaround_exists=True, data_loss=False))

```
{'severity': 'SEV2', 'page_oncall': True, 'response_sla_minutes': 60}
{'severity': 'SEV1', 'page_oncall': True, 'response_sla_minutes': 15}
{'severity': 'SEV2', 'page_oncall': True, 'response_sla_minutes': 60}
```

### 2.2 An incident timeline as structured data

In [ ]:
from datetime import datetime, timezone, timedelta

def build_incident_timeline(events: list[dict]) -> list[dict]:
    """
    events: list of {"time": datetime, "actor": str, "action": str}
    Returns the timeline sorted chronologically with elapsed time
    since the first event added to each entry.
    """
    sorted_events = sorted(events, key=lambda e: e["time"])
    start = sorted_events[0]["time"]
    timeline = []
    for e in sorted_events:
        elapsed = e["time"] - start
        timeline.append({
            "time": e["time"].isoformat(),
            "elapsed_minutes": int(elapsed.total_seconds() // 60),
            "actor": e["actor"],
            "action": e["action"],
        })
    return timeline


base = datetime(2026, 5, 22, 2, 0, tzinfo=timezone.utc)
events = [
    {"time": base + timedelta(minutes=47), "actor": "PagerDuty", "action": "Alert fired: watch_events row count 0 for 8h"},
    {"time": base, "actor": "system", "action": "Pipeline run completes 'successfully' with 0 rows loaded"},
    {"time": base + timedelta(minutes=52), "actor": "you (on-call)", "action": "Acknowledged alert"},
    {"time": base + timedelta(minutes=70), "actor": "you (on-call)", "action": "Identified upstream API schema change as cause"},
    {"time": base + timedelta(minutes=95), "actor": "Carlos", "action": "Confirmed upstream deploy at 01:50 UTC changed field name"},
    {"time": base + timedelta(minutes=130), "actor": "you (on-call)", "action": "Deployed hotfix mapping old->new field name, backfilled 8h gap"},
]

timeline = build_incident_timeline(events)
for entry in timeline:
    print(f"+{entry['elapsed_minutes']:>4}min  {entry['actor']:<14} {entry['action']}")

```
+   0min  system         Pipeline run completes 'successfully' with 0 rows loaded
+  47min  PagerDuty      Alert fired: watch_events row count 0 for 8h
+  52min  you (on-call)  Acknowledged alert
+  70min  you (on-call)  Identified upstream API schema change as cause
+  95min  Carlos         Confirmed upstream deploy at 01:50 UTC changed field name
+ 130min  you (on-call)  Deployed hotfix mapping old->new field name, backfilled 8h gap
```

### 2.3 A blameless postmortem template as a function

In [ ]:
def render_postmortem(
    title: str,
    severity: str,
    detected_at: str,
    resolved_at: str,
    impact: str,
    timeline: list[dict],
    root_cause: str,
    what_went_well: list[str],
    what_went_poorly: list[str],
    action_items: list[dict],
) -> str:
    """
    Render a blameless postmortem as markdown.
    action_items: list of {"action": str, "owner": str, "due": str}
    """
    lines = [
        f"# Postmortem: {title}",
        "",
        f"**Severity:** {severity}  ",
        f"**Detected:** {detected_at}  ",
        f"**Resolved:** {resolved_at}",
        "",
        "## Impact",
        impact,
        "",
        "## Timeline (UTC)",
    ]
    for e in timeline:
        lines.append(f"- +{e['elapsed_minutes']}min — **{e['actor']}**: {e['action']}")

    lines += [
        "",
        "## Root Cause",
        root_cause,
        "",
        "## What Went Well",
    ]
    lines += [f"- {item}" for item in what_went_well]

    lines += ["", "## What Went Poorly"]
    lines += [f"- {item}" for item in what_went_poorly]

    lines += ["", "## Action Items"]
    for item in action_items:
        lines.append(f"- [ ] {item['action']} (owner: {item['owner']}, due: {item['due']})")

    return "\n".join(lines)


doc = render_postmortem(
    title="watch_events pipeline produced 0 rows for 8 hours (2026-05-22)",
    severity="SEV2",
    detected_at="2026-05-22 02:47 UTC",
    resolved_at="2026-05-22 04:10 UTC",
    impact="All country dashboards showed stale (previous-day) watch data for ~8 hours during APAC business hours.",
    timeline=timeline,
    root_cause=(
        "An upstream API deploy at 01:50 UTC renamed the field 'watch_minutes' to "
        "'duration_seconds' without notice. The extract step's schema validation "
        "(Ch 54 contract) treated the missing field as 'no new data' rather than "
        "raising an error, so the pipeline reported success with 0 rows."
    ),
    what_went_well=[
        "On-call acknowledged within 5 minutes of the page",
        "Root cause was identified within 23 minutes of acknowledgment",
        "Hotfix + backfill restored the gap within 2.2 hours total",
    ],
    what_went_poorly=[
        "Detection took 8 hours because the row-count alert threshold was '0 rows for 8h', "
        "tuned for a daily pipeline schedule even though this pipeline runs hourly",
        "The contract validator (Ch 54) silently treated a missing required field as "
        "'zero new rows' instead of failing loudly — this is the Schema-as-Documentation "
        "anti-pattern in disguise: the contract existed but wasn't enforced as a hard failure",
        "No one on the data team was notified of the upstream API deploy in advance",
    ],
    action_items=[
        {"action": "Lower row-count alert threshold to '0 rows for 2h' for hourly pipelines",
         "owner": "you", "due": "2026-05-26"},
        {"action": "Make contract validator raise (not silently skip) when a required field is missing",
         "owner": "you", "due": "2026-05-29"},
        {"action": "Add data team to #backend-deploys Slack channel for advance notice of API changes",
         "owner": "Carlos", "due": "2026-05-23"},
    ],
)

print(doc)

```
# Postmortem: watch_events pipeline produced 0 rows for 8 hours (2026-05-22)

**Severity:** SEV2  
**Detected:** 2026-05-22 02:47 UTC  
**Resolved:** 2026-05-22 04:10 UTC

## Impact
All country dashboards showed stale (previous-day) watch data for ~8 hours during APAC business hours.

## Timeline (UTC)
- +0min — **system**: Pipeline run completes 'successfully' with 0 rows loaded
- +47min — **PagerDuty**: Alert fired: watch_events row count 0 for 8h
- +52min — **you (on-call)**: Acknowledged alert
- +70min — **you (on-call)**: Identified upstream API schema change as cause
- +95min — **Carlos**: Confirmed upstream deploy at 01:50 UTC changed field name
- +130min — **you (on-call)**: Deployed hotfix mapping old->new field name, backfilled 8h gap

## Root Cause
An upstream API deploy at 01:50 UTC renamed the field 'watch_minutes' to 'duration_seconds' without notice. The extract step's schema validation (Ch 54 contract) treated the missing field as 'no new data' rather than raising an error, so the pipeline reported success with 0 rows.

## What Went Well
- On-call acknowledged within 5 minutes of the page
- Root cause was identified within 23 minutes of acknowledgment
- Hotfix + backfill restored the gap within 2.2 hours total

## What Went Poorly
- Detection took 8 hours because the row-count alert threshold was '0 rows for 8h', tuned for a daily pipeline schedule even though this pipeline runs hourly
- The contract validator (Ch 54) silently treated a missing required field as 'zero new rows' instead of failing loudly — this is the Schema-as-Documentation anti-pattern in disguise: the contract existed but wasn't enforced as a hard failure
- No one on the data team was notified of the upstream API deploy in advance

## Action Items
- [ ] Lower row-count alert threshold to '0 rows for 2h' for hourly pipelines (owner: you, due: 2026-05-26)
- [ ] Make contract validator raise (not silently skip) when a required field is missing (owner: you, due: 2026-05-29)
- [ ] Add data team to #backend-deploys Slack channel for advance notice of API changes (owner: Carlos, due: 2026-05-23)
```

### 2.4 Tracking on-call rotation fairness

In [ ]:
from collections import Counter

def summarize_oncall_load(shifts: list[dict]) -> dict:
    """
    shifts: list of {"person": str, "week": str, "pages_received": int}
    Returns per-person totals: number of shifts and total pages.
    """
    shift_counts = Counter()
    page_counts = Counter()
    for s in shifts:
        shift_counts[s["person"]] += 1
        page_counts[s["person"]] += s["pages_received"]

    return {
        "shifts_per_person": dict(shift_counts),
        "pages_per_person": dict(page_counts),
    }


shifts = [
    {"person": "you", "week": "2026-W18", "pages_received": 1},
    {"person": "Carlos", "week": "2026-W19", "pages_received": 0},
    {"person": "you", "week": "2026-W20", "pages_received": 3},
    {"person": "Mei", "week": "2026-W21", "pages_received": 0},
    {"person": "you", "week": "2026-W22", "pages_received": 2},
]

print(summarize_oncall_load(shifts))

```
{'shifts_per_person': {'you': 3, 'Carlos': 1, 'Mei': 1}, 'pages_per_person': {'you': 6, 'Carlos': 0, 'Mei': 0}}
```

## 3. CinemaStream in Practice

In [ ]:
expected_fields = {"user_id", "movie_id", "watch_started", "watch_minutes", "completed", "device", "country"}
actual_response_sample = {
    "user_id": 1,
    "movie_id": 101,
    "watch_started": "2026-05-22T01:55:00Z",
    "duration_seconds": 7920,   # <-- new field name, was 'watch_minutes'
    "completed": True,
    "device": "TV",
    "country": "IN",
}

actual_fields = set(actual_response_sample.keys())
missing = expected_fields - actual_fields
added = actual_fields - expected_fields

print("Missing fields:", missing)
print("New/unexpected fields:", added)

```
Missing fields: {'watch_minutes'}
New/unexpected fields: {'duration_seconds'}
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
result = classify_severity(users_affected_pct=0.5, revenue_impact=False,
                            workaround_exists=False, data_loss=False)
print(result)

```
{'severity': 'SEV3', 'page_oncall': False, 'response_sla_minutes': 240}
```

In [ ]:
from collections import defaultdict

def summarize_incidents_by_cause(incidents: list[dict]) -> dict:
    """
    incidents: list of {"root_cause_category": str, "hours_to_resolve": float}
    Returns per-category: count and total hours lost.
    """
    summary = defaultdict(lambda: {"count": 0, "total_hours": 0.0})
    for inc in incidents:
        cat = inc["root_cause_category"]
        summary[cat]["count"] += 1
        summary[cat]["total_hours"] += inc["hours_to_resolve"]
    return dict(summary)


incidents = [
    {"root_cause_category": "unannounced upstream field change", "hours_to_resolve": 2.2},
    {"root_cause_category": "unannounced upstream field change", "hours_to_resolve": 3.1},
    {"root_cause_category": "unannounced upstream field change", "hours_to_resolve": 2.8},
    {"root_cause_category": "scheduled job timeout", "hours_to_resolve": 0.5},
]

summary = summarize_incidents_by_cause(incidents)
for cat, stats in summary.items():
    print(f"{cat}: {stats['count']} incidents, {stats['total_hours']:.1f} hours total")

```
unannounced upstream field change: 3 incidents, 8.1 hours total
scheduled job timeout: 1 incidents, 0.5 hours total
```

---

# Chapter 58: Real-Time Architecture — Kafka, Kinesis, and Streaming Foundations

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart LR
    P1["Producer\n(watch_events app)"] --> T
    P2["Producer\n(payment service)"] --> T2

    subgraph T["Topic: watch_events\n(retained 7 days)"]
        PA["Partition 0"]
        PB["Partition 1"]
        PC["Partition 2"]
    end

    subgraph T2["Topic: payments"]
        PD["Partition 0"]
    end

    T -->|"Consumer Group A\n(real-time dashboard)"| CGA["Consumer A1\nConsumer A2\nConsumer A3\none per partition"]
    T -->|"Consumer Group B\n(warehouse loader)"| CGB["Consumer B1\n(reads all 3 partitions)\nindependent offset"]
    T2 --> CGB
</div>
"""))

## 2. Theory & Mechanics

### 2.1 A minimal topic: producer writes, consumer reads

In [ ]:
class Topic:
    """A minimal in-memory model of a single-partition Kafka/Kinesis topic."""

    def __init__(self, name: str):
        self.name = name
        self.messages = []  # append-only log

    def produce(self, message: dict):
        offset = len(self.messages)
        self.messages.append({"offset": offset, "value": message})
        return offset

    def read_from(self, offset: int):
        """Return all messages at or after the given offset."""
        return [m for m in self.messages if m["offset"] >= offset]


# A producer writes three watch events as they happen
topic = Topic("watch_events")
topic.produce({"user_id": 1, "movie_id": 101, "event": "play"})
topic.produce({"user_id": 1, "movie_id": 101, "event": "pause"})
topic.produce({"user_id": 2, "movie_id": 103, "event": "play"})

# A consumer starts reading from the beginning (offset 0)
for msg in topic.read_from(0):
    print(msg)

```
{'offset': 0, 'value': {'user_id': 1, 'movie_id': 101, 'event': 'play'}}
{'offset': 1, 'value': {'user_id': 1, 'movie_id': 101, 'event': 'pause'}}
{'offset': 2, 'value': {'user_id': 2, 'movie_id': 103, 'event': 'play'}}
```

### 2.2 Partitions: parallelism with a catch

In [ ]:
def assign_partition(key: str, num_partitions: int) -> int:
    """Deterministically assign a key to a partition (simplified hash-based routing)."""
    return hash(key) % num_partitions


class PartitionedTopic:
    def __init__(self, name: str, num_partitions: int):
        self.name = name
        self.num_partitions = num_partitions
        self.partitions = [[] for _ in range(num_partitions)]

    def produce(self, key: str, message: dict):
        p = assign_partition(key, self.num_partitions)
        offset = len(self.partitions[p])
        self.partitions[p].append({"offset": offset, "value": message})
        return p, offset


topic = PartitionedTopic("watch_events", num_partitions=3)

events = [
    {"user_id": 1, "movie_id": 101, "event": "play"},
    {"user_id": 1, "movie_id": 101, "event": "pause"},
    {"user_id": 2, "movie_id": 103, "event": "play"},
    {"user_id": 1, "movie_id": 102, "event": "play"},
]

for e in events:
    p, offset = topic.produce(key=str(e["user_id"]), message=e)
    print(f"user_id={e['user_id']:<2} -> partition {p}, offset {offset}")

print("\nPartition contents:")
for i, partition in enumerate(topic.partitions):
    print(f"  Partition {i}: {[m['value']['event'] for m in partition]}")

```
user_id=1  -> partition 2, offset 0
user_id=1  -> partition 2, offset 1
user_id=2  -> partition 1, offset 0
user_id=1  -> partition 2, offset 2

Partition contents:
  Partition 0: []
  Partition 1: ['play']
  Partition 2: ['play', 'pause', 'play']
```

### 2.3 Consumer groups: independent readers, shared work

In [ ]:
class ConsumerGroup:
    """Tracks per-partition offsets for a group of consumers reading a topic together."""

    def __init__(self, group_id: str, topic: PartitionedTopic):
        self.group_id = group_id
        self.topic = topic
        # one offset per partition, shared across the group
        self.offsets = [0] * topic.num_partitions

    def poll(self):
        """Return new messages across all partitions since last poll, advance offsets."""
        results = []
        for p, partition in enumerate(self.topic.partitions):
            new_messages = partition[self.offsets[p]:]
            results.extend(new_messages)
            self.offsets[p] = len(partition)
        return results


topic = PartitionedTopic("watch_events", num_partitions=3)
for e in events:
    topic.produce(key=str(e["user_id"]), message=e)

# Two independent consumer groups reading the same topic
realtime_dashboard = ConsumerGroup("realtime-dashboard", topic)
warehouse_loader = ConsumerGroup("warehouse-loader", topic)

print("realtime-dashboard sees:", [m["value"]["event"] for m in realtime_dashboard.poll()])
print("warehouse-loader sees:  ", [m["value"]["event"] for m in warehouse_loader.poll()])

# More events arrive
topic.produce(key="3", message={"user_id": 3, "movie_id": 101, "event": "play"})

print("\nAfter new events:")
print("realtime-dashboard sees:", [m["value"]["event"] for m in realtime_dashboard.poll()])
print("warehouse-loader sees:  ", [m["value"]["event"] for m in warehouse_loader.poll()])
print("realtime-dashboard offsets:", realtime_dashboard.offsets)
print("warehouse-loader offsets:  ", warehouse_loader.offsets)

```
realtime-dashboard sees: ['play', 'play', 'pause', 'play']
warehouse-loader sees:   ['play', 'play', 'pause', 'play']

After new events:
realtime-dashboard sees: ['play']
warehouse-loader sees:   ['play']
realtime-dashboard offsets: [0, 2, 3]
warehouse-loader offsets:   [0, 2, 3]
```

### 2.4 At-least-once delivery and idempotency (again)

In [ ]:
def process_watch_event_idempotent(event: dict, processed_event_ids: set) -> dict:
    """
    Process a watch event exactly-once-in-effect, even under at-least-once delivery,
    by tracking which event_ids have already been applied.
    """
    event_id = event["event_id"]
    if event_id in processed_event_ids:
        return {"status": "skipped_duplicate", "event_id": event_id}

    # ... apply the event (e.g., update a running "currently watching" count) ...
    processed_event_ids.add(event_id)
    return {"status": "applied", "event_id": event_id}


processed = set()
event = {"event_id": "evt-9001", "user_id": 1, "movie_id": 101, "event": "play"}

# First delivery
print(process_watch_event_idempotent(event, processed))
# Consumer crashes here before committing offset, restarts, message redelivered
print(process_watch_event_idempotent(event, processed))

```
{'status': 'applied', 'event_id': 'evt-9001'}
{'status': 'skipped_duplicate', 'event_id': 'evt-9001'}
```

## 3. CinemaStream in Practice

In [ ]:
from collections import defaultdict, deque
from datetime import datetime, timedelta, timezone

def compute_trending_now(events: list[dict], window_minutes: int = 5, now: datetime = None) -> list[tuple]:
    """
    Given a stream of 'play' events with timestamps, return movie_ids ranked by
    play count within the last `window_minutes` of `now`.
    """
    if now is None:
        now = max(e["watch_started"] for e in events)
    cutoff = now - timedelta(minutes=window_minutes)

    counts = defaultdict(int)
    for e in events:
        if e["event"] == "play" and cutoff <= e["watch_started"] <= now:
            counts[e["movie_id"]] += 1

    return sorted(counts.items(), key=lambda x: x[1], reverse=True)


now = datetime(2026, 6, 10, 12, 0, tzinfo=timezone.utc)
stream_events = [
    {"movie_id": 101, "event": "play", "watch_started": now - timedelta(minutes=1)},
    {"movie_id": 101, "event": "play", "watch_started": now - timedelta(minutes=2)},
    {"movie_id": 102, "event": "play", "watch_started": now - timedelta(minutes=3)},
    {"movie_id": 103, "event": "play", "watch_started": now - timedelta(minutes=8)},  # outside window
    {"movie_id": 101, "event": "play", "watch_started": now - timedelta(seconds=30)},
]

trending = compute_trending_now(stream_events, window_minutes=5, now=now)
print("Trending Now (movie_id, plays in last 5 min):")
for movie_id, count in trending:
    print(f"  movie_id={movie_id}: {count} plays")

```
Trending Now (movie_id, plays in last 5 min):
  movie_id=101: 3 plays
  movie_id=102: 1 plays
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
topic = PartitionedTopic("test_topic", num_partitions=2)
for i in range(5):
    p, offset = topic.produce(key="same-key", message={"seq": i})
    print(f"message seq={i} -> partition {p}, offset {offset}")

group = ConsumerGroup("test-group", topic)
results = group.poll()
print("\nConsumed order:", [m["value"]["seq"] for m in results])

```
message seq=0 -> partition 0, offset 0
message seq=1 -> partition 0, offset 1
message seq=2 -> partition 0, offset 2
message seq=3 -> partition 0, offset 3
message seq=4 -> partition 0, offset 4

Consumed order: [0, 1, 2, 3, 4]
```

In [ ]:
def compute_consumer_lag(partition_sizes: list[int], consumer_offsets: list[int]) -> dict:
    """
    Lag = how many messages in each partition the consumer hasn't read yet.
    """
    if len(partition_sizes) != len(consumer_offsets):
        raise ValueError("partition_sizes and consumer_offsets must have the same length")

    per_partition_lag = [size - offset for size, offset in zip(partition_sizes, consumer_offsets)]
    return {
        "per_partition_lag": per_partition_lag,
        "total_lag": sum(per_partition_lag),
    }


partition_sizes = [15000, 15200, 14800]
consumer_offsets = [15000, 12100, 14800]

result = compute_consumer_lag(partition_sizes, consumer_offsets)
print(result)

```
{'per_partition_lag': [0, 3100, 0], 'total_lag': 3100}
```

---

# Chapter 59: Data Mesh & Decentralized Ownership

## 0. Where You Are

## 1. The Concept

In [ ]:
from IPython.display import HTML as _HTML, display as _display
_display(_HTML("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true, theme:'neutral'});</script>
<div class="mermaid">
flowchart TD
    subgraph PLATFORM["Self-Serve Platform (central team)"]
        INFRA[Storage + Compute\nCatalog + Access Control]
        GOV[Federated Governance\nNaming · PII · SLAs]
    end
    subgraph DOMAINS["Domain Teams (own their data)"]
        BE[Backend team\nwatch_events product]
        BILL[Billing team\nsubscriptions product]
        SUP[Support team\ntickets product]
    end
    PLATFORM --> DOMAINS
    BE -->|data contract| CONS[Consumers\nanalytics · ML · finance]
    BILL -->|data contract| CONS
    SUP -->|data contract| CONS
</div>
"""))

## 2. Theory & Mechanics

### 2.1 Modeling ownership: who is responsible for what?

In [ ]:
data_products = [
    {"name": "watch_events", "domain": "content_platform", "owner_team": "backend",
     "owner_contact": "carlos", "sla_freshness_minutes": 60, "quality_checks": True},
    {"name": "subscriptions", "domain": "billing", "owner_team": "billing",
     "owner_contact": "billing-team-lead", "sla_freshness_minutes": 1440, "quality_checks": True},
    {"name": "support_tickets", "domain": "support", "owner_team": "support",
     "owner_contact": "support-ops-lead", "sla_freshness_minutes": 1440, "quality_checks": False},
    {"name": "ratings", "domain": "content_platform", "owner_team": "backend",
     "owner_contact": "carlos", "sla_freshness_minutes": 1440, "quality_checks": True},
]


def find_data_products_without_quality_checks(products: list[dict]) -> list[str]:
    """Return names of data products that don't have automated quality checks (Ch 53/54)."""
    return [p["name"] for p in products if not p["quality_checks"]]


def find_owner(products: list[dict], product_name: str) -> dict | None:
    """Find the owning team and contact for a given data product."""
    for p in products:
        if p["name"] == product_name:
            return {"owner_team": p["owner_team"], "owner_contact": p["owner_contact"]}
    return None


print("Products without quality checks:", find_data_products_without_quality_checks(data_products))
print("Who owns 'support_tickets'?", find_owner(data_products, "support_tickets"))
print("Who owns 'nonexistent_table'?", find_owner(data_products, "nonexistent_table"))

```
Products without quality checks: ['support_tickets']
Who owns 'support_tickets'? {'owner_team': 'support', 'owner_contact': 'support-ops-lead'}
Who owns 'nonexistent_table'? None
```

### 2.2 A minimal data product "contract" — extending Chapter 54

In [ ]:
def render_data_product_card(product: dict, schema_fields: list[dict]) -> str:
    """
    Render a 'data product card' — a short, standard-format description any
    consumer can read to understand a data product without asking its owner.
    """
    lines = [
        f"# Data Product: {product['name']}",
        f"**Domain:** {product['domain']}",
        f"**Owner team:** {product['owner_team']} (contact: {product['owner_contact']})",
        f"**Freshness SLA:** updated at least every {product['sla_freshness_minutes']} minutes",
        f"**Automated quality checks:** {'yes' if product['quality_checks'] else 'NO — use with caution'}",
        "",
        "## Schema",
    ]
    for field in schema_fields:
        lines.append(f"- `{field['name']}` ({field['type']}){' — ' + field['note'] if field.get('note') else ''}")
    return "\n".join(lines)


support_tickets_schema = [
    {"name": "ticket_id", "type": "INT", "note": "primary key"},
    {"name": "user_id", "type": "INT", "note": "foreign key -> users.user_id"},
    {"name": "created_at", "type": "TIMESTAMP", "note": "UTC"},
    {"name": "category", "type": "TEXT"},
    {"name": "priority", "type": "TEXT"},
    {"name": "status", "type": "TEXT"},
]

card = render_data_product_card(data_products[2], support_tickets_schema)
print(card)

```
# Data Product: support_tickets
**Domain:** support
**Owner team:** support (contact: support-ops-lead)
**Freshness SLA:** updated at least every 1440 minutes
**Automated quality checks:** NO — use with caution

## Schema
- `ticket_id` (INT) — primary key
- `user_id` (INT) — foreign key -> users.user_id
- `created_at` (TIMESTAMP) — UTC
- `category` (TEXT)
- `priority` (TEXT)
- `status` (TEXT)
```

### 2.3 Federated governance as automated checks

In [ ]:
def check_federated_standards(product: dict, schema_fields: list[dict]) -> list[str]:
    """
    Run global standards checks against a data product, regardless of owning team.
    Returns a list of violations (empty list = compliant).
    """
    violations = []

    # Standard 1: every product must declare an owner contact
    if not product.get("owner_contact"):
        violations.append("Missing owner_contact")

    # Standard 2: every table must have a documented primary key
    pk_fields = [f for f in schema_fields if f.get("note", "").startswith("primary key")]
    if len(pk_fields) != 1:
        violations.append(f"Expected exactly 1 primary key field, found {len(pk_fields)}")

    # Standard 3: any TIMESTAMP field must document its timezone
    for f in schema_fields:
        if f["type"] == "TIMESTAMP" and "UTC" not in f.get("note", ""):
            violations.append(f"Field '{f['name']}' is TIMESTAMP but doesn't document timezone (must be UTC)")

    # Standard 4: freshness SLA must be set (not zero/missing)
    if not product.get("sla_freshness_minutes"):
        violations.append("Missing or zero sla_freshness_minutes")

    return violations


# A compliant product
print("support_tickets:", check_federated_standards(data_products[2], support_tickets_schema))

# A non-compliant product: timestamp without timezone documented, no PK marked
recommendations_schema = [
    {"name": "rec_id", "type": "INT"},  # not marked as primary key
    {"name": "user_id", "type": "INT"},
    {"name": "generated_at", "type": "TIMESTAMP"},  # no timezone note
]
recommendations_product = {"name": "recommendations", "domain": "ml", "owner_team": "ml",
                            "owner_contact": "mei", "sla_freshness_minutes": 60, "quality_checks": False}

print("recommendations:", check_federated_standards(recommendations_product, recommendations_schema))

```
support_tickets: []
recommendations: ['Expected exactly 1 primary key field, found 0', "Field 'generated_at' is TIMESTAMP but doesn't document timezone (must be UTC)"]
```

## 3. CinemaStream in Practice

In [ ]:
data_products_extended = [
    {"name": "watch_events", "domain": "content_platform", "owner_team": "data_team",
     "owner_contact": "you", "sla_freshness_minutes": 60, "quality_checks": True,
     "weekly_adhoc_requests": 1},
    {"name": "subscriptions", "domain": "billing", "owner_team": "data_team",
     "owner_contact": "you", "sla_freshness_minutes": 1440, "quality_checks": True,
     "weekly_adhoc_requests": 0},
    {"name": "support_tickets", "domain": "support", "owner_team": "data_team",
     "owner_contact": "you", "sla_freshness_minutes": 1440, "quality_checks": False,
     "weekly_adhoc_requests": 4},
    {"name": "ml_churn_features", "domain": "ml", "owner_team": "data_team",
     "owner_contact": "you", "sla_freshness_minutes": 1440, "quality_checks": False,
     "weekly_adhoc_requests": 5},
]


def recommend_ownership_changes(products: list[dict], threshold: int = 2) -> list[dict]:
    """
    Recommend which data products are candidates for domain ownership:
    high ad-hoc request volume from a single non-data team suggests that
    team would benefit from (and is motivated to invest in) owning the pipeline.
    """
    recommendations = []
    for p in products:
        if p["weekly_adhoc_requests"] >= threshold and p["owner_team"] == "data_team":
            recommendations.append({
                "product": p["name"],
                "current_owner": p["owner_team"],
                "weekly_adhoc_requests": p["weekly_adhoc_requests"],
                "recommendation": f"Consider transferring ownership of '{p['name']}' to the "
                                   f"'{p['domain']}' domain team — high request volume "
                                   f"({p['weekly_adhoc_requests']}/week) suggests they're "
                                   f"already the primary stakeholder.",
            })
    return recommendations


for rec in recommend_ownership_changes(data_products_extended):
    print(rec["product"], "->", rec["recommendation"])
    print(f"  (weekly ad-hoc requests: {rec['weekly_adhoc_requests']})")

```
support_tickets -> Consider transferring ownership of 'support_tickets' to the 'support' domain team — high request volume (4/week) suggests they're already the primary stakeholder.
  (weekly ad-hoc requests: 4)
ml_churn_features -> Consider transferring ownership of 'ml_churn_features' to the 'ml' domain team — high request volume (5/week) suggests they're already the primary stakeholder.
  (weekly ad-hoc requests: 5)
```

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
orders_product = {"name": "orders", "domain": "commerce", "owner_team": "commerce",
                  "owner_contact": None, "sla_freshness_minutes": 0, "quality_checks": True}

orders_schema = [
    {"name": "order_id", "type": "INT", "note": "primary key"},
    {"name": "order_uuid", "type": "TEXT", "note": "primary key"},  # mistakenly also marked PK
]

violations = check_federated_standards(orders_product, orders_schema)
for v in violations:
    print("-", v)

```
- Missing owner_contact
- Expected exactly 1 primary key field, found 2
- Missing or zero sla_freshness_minutes
```

In [ ]:
support_tickets_schema_v2 = support_tickets_schema + [
    {"name": "resolution_notes", "type": "TEXT"},
]

violations = check_federated_standards(data_products[2], support_tickets_schema_v2)
print(violations)

```
[]
```